# 02 · Procesamiento InSAR multifrecuencia / *Multi-frequency InSAR processing*
**Sentinel-1 (C) · NISAR (L) · ALOS-2 (L)** — Proyecto / *Project* **GUAIRA-SAR** · SRL *2026 Venezuela, Colombia and Panama Earthquakes*

| # | 🇪🇸 Flujo común para los tres sensores | 🇬🇧 Common workflow for the three sensors |
|---|---|---|
| 1 | Fase desenrollada → LOS en metros, **mismo signo** (+ = hacia el satélite) | Unwrapped phase → LOS in metres, **same sign** (+ = toward satellite) |
| 2 | Máscara por coherencia ≥ 0,3 | Coherence mask ≥ 0.3 |
| 3 | Corrección ionosférica en banda L (NISAR) | L-band ionospheric correction (NISAR) |
| 4 | Grilla común UTM 19N, 80 m | Common UTM 19N grid, 80 m |
| 5 | Descomposición Este / Vertical (asc + desc, N ≈ 0) | East / Up decomposition (asc + desc, N ≈ 0) |
| 6 | Validación GNSS CCS1 y comparación C vs L | GNSS CCS1 validation and C vs L comparison |

🇪🇸 Corra en orden; si Colab se reinicia, empiece por la Celda 1. 🇬🇧 Run in order; if Colab restarts, start from Cell 1.

## Celda 1 · Librerías, Drive y carpetas / *Cell 1 · Libraries, Drive and folders*

In [ ]:
!pip install -q asf_search rasterio h5py pyproj shapely pandas matplotlib requests
from google.colab import drive
import os
drive.mount('/content/drive')
RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
for d in ["01_datos_crudos/auxiliares", "02_procesados", "03_resultados", "04_figuras", "05_documentacion"]:
    os.makedirs(f"{RAIZ}/{d}", exist_ok=True)
print("OK:", RAIZ)


## Celda 2 · ¿Qué ALOS-2 hay en ASF? / *Cell 2 · Which ALOS-2 data are on ASF?*
🇪🇸 JAXA liberó el ScanSAR del sismo el 3 de julio de 2026. ASF guarda un espejo del ScanSAR L1.1, pero llega con retraso y cada escena pesa ~28–56 GB.  
🇬🇧 JAXA released the ScanSAR data for this event on 3 July 2026. ASF mirrors ScanSAR L1.1, with some delay; each scene is ~28–56 GB.

In [ ]:
import asf_search as asf, pandas as pd
from shapely.geometry import box, shape
ROI = box(-68.6, 10.0, -66.4, 11.0)
SISMO = pd.Timestamp("2026-06-24T22:04:33Z")
res = []
for kw in ({"dataset": "ALOS-2"}, {"platform": "ALOS-2"}):
    try:
        res = asf.search(intersectsWith=ROI.wkt, start="2025-01-01", end="2026-08-31", **kw)
        if len(res):
            print("Búsqueda OK con", kw); break
    except Exception as e:
        print(kw, "->", e)
filas = [dict(escena=r.properties.get("sceneName"), fecha=pd.Timestamp(r.properties["startTime"]),
              path=r.properties.get("pathNumber"), frame=r.properties.get("frameNumber"),
              dir=r.properties.get("flightDirection"), modo=r.properties.get("beamModeType"),
              GB=round(sum(v.get("bytes", 0) for v in r.properties["bytes"].values()) / 1e9, 1)
                 if isinstance(r.properties.get("bytes"), dict) else None,
              url=r.properties.get("url")) for r in res]
alos2 = pd.DataFrame(filas)
if alos2.empty:
    print("ASF todavía no tiene ALOS-2 de 2026 sobre la zona (el espejo de JAXA llega con retraso).")
    print("Alternativas: (1) JAXA liberó el ScanSAR del sismo el 3-jul-2026: descargar desde el portal de JAXA")
    print("cuando su sitio vuelva; (2) usar los LOS ALOS-2 ya procesados por Scripps/UCSD (topex.ucsd.edu/VNZ_7.5), citándolos.")
else:
    alos2["relacion"] = alos2.fecha.map(lambda f: "PRE" if f < SISMO else "POST")
    alos2 = alos2.sort_values(["path", "fecha"])
    print(alos2.drop(columns="url").to_string(index=False))
    alos2.to_csv(f"{RAIZ}/05_documentacion/alos2_escenas.csv", index=False)


## Celda 3 · Datos auxiliares / *Cell 3 · Ancillary data*
🇪🇸 Réplicas USGS, cobertura del suelo ESA WorldCover. 🇬🇧 USGS aftershocks, ESA WorldCover land cover.

In [ ]:
import requests, os
AUX = f"{RAIZ}/01_datos_crudos/auxiliares"; os.makedirs(AUX, exist_ok=True)

# 1) Réplicas: catálogo USGS ComCat (M>=2.5, desde el sismo hasta hoy)
url = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2026-06-24"
       "&minlatitude=9.8&maxlatitude=11.2&minlongitude=-69.2&maxlongitude=-65.8&minmagnitude=2.5")
open(f"{AUX}/usgs_replicas_2026.csv", "wb").write(requests.get(url, timeout=120).content)
print("Réplicas USGS guardadas")

# 2) Cobertura del suelo ESA WorldCover 2021 (10 m) – tesela N09W069 cubre Yumare..Guatire
wc = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N09W069_Map.tif"
dst = f"{AUX}/ESA_WorldCover_10m_2021_N09W069.tif"
if not os.path.exists(dst):
    with requests.get(wc, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dst, "wb") as f:
            for ch in r.iter_content(1 << 20): f.write(ch)
print("WorldCover guardado")

# 3) Descarga manual (citar con fecha de acceso en Data and Resources):
#    - Copernicus EMS EMSR884 (mapas de daño): https://mapping.emergency.copernicus.eu/activations/EMSR884/
#    - Desplazamientos GNSS cosísmicos del Nevada Geodetic Laboratory: https://geodesy.unr.edu/
#    - Modelo de falla finita y trazas: páginas USGS us6000t7zc y us6000t7zp
#    - LOS ALOS-2 / NISAR procesados por Scripps/UCSD: https://topex.ucsd.edu/VNZ_7.5/


## Celda 4 · NISAR GUNW → GeoTIFF / *Cell 4 · NISAR GUNW → GeoTIFF*
🇪🇸 LOS (m), coherencia, ionosfera, vectores de mirada. 🇬🇧 LOS (m), coherence, ionosphere, look vectors.

In [ ]:
import glob
import json
import os
import re

import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin

ENTRADA = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/01_datos_crudos/nisar_gunw"
SALIDA = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/02_procesados/nisar"
os.makedirs(SALIDA, exist_ok=True)
C = 299_792_458.0

# Convención común del proyecto: LOS positivo = acercamiento al satélite (range decrease).
# Si la validación con GNSS CCS1 (dE ≈ -0.46 m, Suárez & Arenas 2026) sale con signo
# opuesto, cambiar SIGNO a +1. NISAR mira a la IZQUIERDA: en órbita ascendente mira al
# oeste, así que el movimiento al oeste de CCS1 debe verse como ACERCAMIENTO (+).
SIGNO = -1


def buscar(h5, termina, contiene=None):
    hits = []
    h5.visit(lambda n: hits.append(n) if n.endswith(termina) and (contiene is None or contiene in n) else None)
    return hits[0] if hits else None


def guardar(arr, x, y, epsg, ruta, nodata=np.nan):
    dx, dy = float(x[1] - x[0]), float(y[1] - y[0])
    arr = np.asarray(arr, dtype="float32")
    if dy > 0:                      # filas de sur a norte -> voltear
        arr, dy = arr[::-1], -dy
        y = y[::-1]
    tr = from_origin(float(x[0]) - dx / 2, float(y[0]) - dy / 2, dx, -dy)
    with rasterio.open(ruta, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype="float32", crs=f"EPSG:{epsg}", transform=tr,
                       nodata=nodata, compress="deflate", tiled=True) as dst:
        dst.write(arr, 1)


resumen = []
for f in sorted(glob.glob(f"{ENTRADA}/*GUNW*.h5")):
    base = os.path.basename(f).replace(".h5", "")
    m = re.search(r"GUNW_\d+_(\d{3})_([AD])_(\d{3})", base)
    track = f"{m.group(1)}{m.group(2)}" if m else "NA"
    fechas = re.findall(r"(\d{8})T\d{6}", base)
    corto = f"NISAR_T{track}_{fechas[0]}_{fechas[-1]}" if fechas else base[:40]
    print(f"\n== {corto}")
    with h5py.File(f, "r") as h:
        p_unw = buscar(h, "unwrappedPhase", "unwrappedInterferogram")
        p_coh = buscar(h, "coherenceMagnitude", "unwrappedInterferogram")
        p_cc = buscar(h, "connectedComponents", "unwrappedInterferogram")
        p_ion = buscar(h, "ionospherePhaseScreen", "unwrappedInterferogram")
        grupo = p_unw.rsplit("/", 2)[0]                     # .../unwrappedInterferogram
        p_x = buscar(h, "xCoordinates", grupo) or buscar(h, "xCoordinates", "frequencyA")
        p_y = buscar(h, "yCoordinates", grupo) or buscar(h, "yCoordinates", "frequencyA")
        p_proj = buscar(h, "projection", grupo) or buscar(h, "projection", "frequencyA")
        p_f = buscar(h, "centerFrequency", "frequencyA")

        x, y = h[p_x][:], h[p_y][:]
        epsg = int(h[p_proj].attrs.get("epsg_code", h[p_proj][()]))
        f0 = float(np.ravel(h[p_f][()])[0]) if p_f else 1.257e9
        lam = C / f0
        fill = h[p_unw].attrs.get("_FillValue", 0)

        unw = h[p_unw][:].astype("float32")
        unw[(unw == fill) | ~np.isfinite(unw)] = np.nan
        los = SIGNO * unw * lam / (4 * np.pi)
        guardar(los, x, y, epsg, f"{SALIDA}/{corto}_los_m.tif")
        guardar(h[p_coh][:], x, y, epsg, f"{SALIDA}/{corto}_coh.tif")
        if p_cc:
            guardar(h[p_cc][:].astype("float32"), x, y, epsg, f"{SALIDA}/{corto}_concomp.tif", nodata=-1)
        if p_ion:
            ion = h[p_ion][:].astype("float32")
            if ion.shape == unw.shape:
                guardar(SIGNO * (unw - ion) * lam / (4 * np.pi), x, y, epsg,
                        f"{SALIDA}/{corto}_los_iono_m.tif")
            else:
                print(f"   ionosfera en otra grilla {ion.shape} vs {unw.shape}: se guarda aparte")
                xi = h[buscar(h, "xCoordinates", p_ion.rsplit('/', 1)[0])][:]
                yi = h[buscar(h, "yCoordinates", p_ion.rsplit('/', 1)[0])][:]
                guardar(SIGNO * ion * lam / (4 * np.pi), xi, yi, epsg, f"{SALIDA}/{corto}_iono_m.tif")

        # Vectores de mirada (cubo radarGrid: altura × y × x) -> capa más cercana a 0 m
        vec = {}
        for comp in ("X", "Y", "Z"):
            p = buscar(h, f"losUnitVector{comp}", "radarGrid")
            if p:
                vec[comp] = p
        if vec:
            rg = vec["X"].rsplit("/", 1)[0]
            xr, yr = h[buscar(h, "xCoordinates", rg)][:], h[buscar(h, "yCoordinates", rg)][:]
            pz = buscar(h, "heightAboveEllipsoid", rg)
            k = int(np.argmin(np.abs(h[pz][:]))) if pz else 0
            lx, ly = h[vec["X"]][k], h[vec["Y"]][k]
            lz = h[vec["Z"]][k] if "Z" in vec else np.sqrt(np.clip(1 - lx**2 - ly**2, 0, 1))
            # Vector suelo->satélite (E, N, U); con nuestra convención LOS+ = hacia el satélite
            for nombre, comp in (("E", lx), ("N", ly), ("U", lz)):
                guardar(comp, xr, yr, epsg, f"{SALIDA}/{corto}_los{nombre}.tif")
        else:
            print("   [!] No se encontraron vectores LOS en radarGrid")

        v = los[np.isfinite(los)]
        info = dict(producto=base, nombre_corto=corto, track=track, epsg=epsg,
                    lambda_m=round(lam, 5), fecha_ref=fechas[0] if fechas else None,
                    fecha_sec=fechas[-1] if fechas else None,
                    tiene_ionosfera=bool(p_ion), tiene_vectores_los=bool(vec),
                    los_p2_m=float(np.percentile(v, 2)) if v.size else None,
                    los_p98_m=float(np.percentile(v, 98)) if v.size else None,
                    coh_media=float(np.nanmean(h[p_coh][:])))
        resumen.append(info)
        print(json.dumps(info, indent=1))

json.dump(resumen, open(f"{SALIDA}/resumen_nisar.json", "w"), indent=1)
print(f"\n{len(resumen)} productos convertidos -> {SALIDA}")


## Celda 5 · Sentinel-1 HyP3 → GeoTIFF / *Cell 5 · Sentinel-1 HyP3 → GeoTIFF*
🇪🇸 Requiere los trabajos HyP3 descargados (Celda 11 del notebook 01). 🇬🇧 Requires the HyP3 jobs downloaded (Cell 11 of notebook 01).

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# BLOQUE A · Sentinel-1 HyP3 (INSAR_GAMMA) -> GeoTIFF con la convención común
# =============================================================================
import glob, os, zipfile, json
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import from_origin

RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
S1_IN = f"{RAIZ}/01_datos_crudos/sentinel1_hyp3"
S1_OUT = f"{RAIZ}/02_procesados/sentinel1"
os.makedirs(S1_OUT, exist_ok=True)

LAMBDA_S1 = 0.055465763          # m (5.405 GHz)
SIGNO_S1 = -1                    # + = acercamiento al satélite; verificar con GNSS CCS1

for z in glob.glob(f"{S1_IN}/*.zip"):
    destino = z[:-4]
    if not os.path.isdir(destino):
        zipfile.ZipFile(z).extractall(os.path.dirname(z))

for carpeta in sorted(glob.glob(f"{S1_IN}/S1*_INT*")):
    if not os.path.isdir(carpeta):
        continue
    unw = glob.glob(f"{carpeta}/*_unw_phase.tif")
    if not unw:
        continue
    base = os.path.basename(carpeta)
    fechas = [t[:8] for t in base.split("_") if len(t) == 15 and t[8] == "T"]
    # track desde los metadatos (*.txt de HyP3)
    meta = {}
    for t in glob.glob(f"{carpeta}/*.txt"):
        for linea in open(t, errors="ignore"):
            if ":" in linea:
                k, v = linea.split(":", 1)
                meta[k.strip()] = v.strip()
    nombre = f"S1_{fechas[0]}_{fechas[-1]}_{base[-4:]}" if fechas else base
    with rasterio.open(unw[0]) as s:
        fase = s.read(1).astype("float32"); perfil = s.profile
    fase[fase == 0] = np.nan
    perfil.update(dtype="float32", nodata=np.nan, compress="deflate")
    with rasterio.open(f"{S1_OUT}/{nombre}_los_m.tif", "w", **perfil) as d:
        d.write(SIGNO_S1 * fase * LAMBDA_S1 / (4 * np.pi), 1)
    for suf, sal in (("_corr.tif", "_coh.tif"),):
        f = glob.glob(f"{carpeta}/*{suf}")
        if f:
            with rasterio.open(f[0]) as s:
                with rasterio.open(f"{S1_OUT}/{nombre}{sal}", "w", **perfil) as d:
                    d.write(s.read(1).astype("float32"), 1)
    # Vectores de mirada HyP3: lv_theta (elevación desde horizontal), lv_phi (desde el Este, antihorario)
    th, ph = glob.glob(f"{carpeta}/*_lv_theta.tif"), glob.glob(f"{carpeta}/*_lv_phi.tif")
    if th and ph:
        with rasterio.open(th[0]) as s: theta = s.read(1).astype("float32")
        with rasterio.open(ph[0]) as s: phi = s.read(1).astype("float32")
        theta[theta == 0] = np.nan
        comps = {"E": np.cos(theta) * np.cos(phi), "N": np.cos(theta) * np.sin(phi), "U": np.sin(theta)}
        for k, v in comps.items():
            with rasterio.open(f"{S1_OUT}/{nombre}_los{k}.tif", "w", **perfil) as d:
                d.write(v.astype("float32"), 1)
    else:
        print(f"[!] {nombre}: sin lv_theta/lv_phi (reenviar a HyP3 con include_look_vectors=True)")
    print("✓", nombre)


## Celda 6 · Grilla común y descomposición Este/Vertical / *Cell 6 · Common grid and East/Up decomposition*

In [ ]:
# =============================================================================
# BLOQUE B · Descomposición Este/Vertical (asc + desc) en una grilla común
#            Se aplica IGUAL a Sentinel-1, NISAR y ALOS-2
# =============================================================================
GRILLA_EPSG = 32619
RES = 80.0                                   # m (resolución común de comparación)
XMIN, XMAX, YMIN, YMAX = 640000, 790000, 1140000, 1200000   # UTM 19N: Yumare..Guatire
COH_MIN = 0.3


def a_grilla(ruta, metodo=Resampling.bilinear):
    n_x, n_y = int((XMAX - XMIN) / RES), int((YMAX - YMIN) / RES)
    dst = np.full((n_y, n_x), np.nan, dtype="float32")
    tr = from_origin(XMIN, YMAX, RES, RES)
    with rasterio.open(ruta) as s:
        reproject(rasterio.band(s, 1), dst, src_transform=s.transform, src_crs=s.crs,
                  src_nodata=s.nodata, dst_transform=tr, dst_crs=f"EPSG:{GRILLA_EPSG}",
                  dst_nodata=np.nan, resampling=metodo)
    return dst, tr


def descomponer(pares, nombre_banda, salida):
    """pares: lista de prefijos (asc y desc) con archivos *_los_m, *_coh, *_losE/N/U.
    Resuelve por mínimos cuadrados d_LOS = E·e + U·u (N≈0; GNSS CCS1 da dN≈-1 cm)."""
    os.makedirs(salida, exist_ok=True)
    G, d = [], []
    for pref in pares:
        los, tr = a_grilla(f"{pref}_los_m.tif")
        coh, _ = a_grilla(f"{pref}_coh.tif")
        e, _ = a_grilla(f"{pref}_losE.tif"); u, _ = a_grilla(f"{pref}_losU.tif")
        los[coh < COH_MIN] = np.nan
        G.append(np.stack([e, u], -1)); d.append(los)
    G = np.stack(G, -2)                       # (y, x, n_obs, 2)
    d = np.stack(d, -1)                       # (y, x, n_obs)
    ok = np.all(np.isfinite(d), -1) & np.all(np.isfinite(G), (-1, -2))
    E = np.full(ok.shape, np.nan, "float32"); U = E.copy()
    Gv, dv = G[ok], d[ok][..., None]
    sol = np.linalg.pinv(Gv) @ dv             # (n, 2, 1)
    E[ok], U[ok] = sol[:, 0, 0], sol[:, 1, 0]
    perfil = dict(driver="GTiff", height=E.shape[0], width=E.shape[1], count=1, dtype="float32",
                  crs=f"EPSG:{GRILLA_EPSG}", transform=tr, nodata=np.nan, compress="deflate")
    for k, v in (("E", E), ("U", U)):
        with rasterio.open(f"{salida}/{nombre_banda}_{k}_m.tif", "w", **perfil) as dst:
            dst.write(v, 1)
    print(f"{nombre_banda}: {ok.sum()} píxeles resueltos | E p2-p98 = "
          f"{np.nanpercentile(E, 2):.2f}..{np.nanpercentile(E, 98):.2f} m")
    return E, U, tr


def valor_en(arr, tr, x, y, ventana=2):
    c, f = int((x - tr.c) / tr.a), int((y - tr.f) / tr.e)
    return np.nanmedian(arr[f - ventana:f + ventana + 1, c - ventana:c + ventana + 1])


## Celda 7 · Descomponer NISAR y Sentinel-1 + validación GNSS / *Cell 7 · Decompose NISAR and Sentinel-1 + GNSS validation*

In [ ]:
import glob
# --- NISAR: 2 ascendentes (061A, 162A) + 1 descendente (126D)
pref_n = sorted({p.rsplit("_los_m.tif", 1)[0] for p in glob.glob(f"{RAIZ}/02_procesados/nisar/*_los_m.tif")
                 if any(t in p for t in ("T061A", "T162A", "T126D"))})
print("NISAR:", [os.path.basename(p) for p in pref_n])
E_L, U_L, tr = descomponer(pref_n, "NISAR_L", f"{RAIZ}/02_procesados/descomposicion")

# --- Sentinel-1: 033A + 025D
pref_c = sorted({p.rsplit("_los_m.tif", 1)[0] for p in glob.glob(f"{RAIZ}/02_procesados/sentinel1/*_los_m.tif")})
print("Sentinel-1:", [os.path.basename(p) for p in pref_c])
E_C, U_C, _ = descomponer(pref_c, "S1_C", f"{RAIZ}/02_procesados/descomposicion")

# --- Validación con GNSS CCS1 (dE = -0.464 m, dN = -0.011 m, dU = +0.021 m; Arenas & Suárez 2026)
# Coloque las coordenadas exactas de CCS1 (de la ficha de estación NGL o IGVSB):
CCS1_LON, CCS1_LAT = None, None
if CCS1_LON is not None:
    from pyproj import Transformer
    x, y = Transformer.from_crs(4326, GRILLA_EPSG, always_xy=True).transform(CCS1_LON, CCS1_LAT)
    for nombre, E, U in (("NISAR L", E_L, U_L), ("Sentinel-1 C", E_C, U_C)):
        print(f"{nombre}: E = {valor_en(E, tr, x, y):+.3f} m (GNSS -0.464) | U = {valor_en(U, tr, x, y):+.3f} m (GNSS +0.021)")
    print("Si E sale POSITIVO (~+0.46), cambie SIGNO / SIGNO_S1 a +1 y vuelva a correr las celdas 4 y 5.")


## Celda 8 · Borrador: Este y Vertical, banda C vs L / *Cell 8 · Draft figure: East and Up, C vs L band*

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
ext = [XMIN/1e3, XMAX/1e3, YMIN/1e3, YMAX/1e3]
fig, axs = plt.subplots(2, 3, figsize=(16, 6.5), constrained_layout=True)
paneles = [(E_C, "Sentinel-1 (C) · Este"), (E_L, "NISAR (L) · Este"), (E_L - E_C, "L − C · Este"),
           (U_C, "Sentinel-1 (C) · Vertical"), (U_L, "NISAR (L) · Vertical"), (U_L - U_C, "L − C · Vertical")]
for ax, (a, t) in zip(axs.flat, paneles):
    lim = np.nanpercentile(np.abs(a), 98) if np.isfinite(a).any() else 1
    im = ax.imshow(a, extent=ext, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim))
    ax.set_title(t, loc="left", fontsize=10); ax.set_xlabel("Este UTM 19N (km)"); ax.set_ylabel("Norte (km)")
    fig.colorbar(im, ax=ax, shrink=0.8, label="m")
os.makedirs(f"{RAIZ}/04_figuras/borradores", exist_ok=True)
plt.savefig(f"{RAIZ}/04_figuras/borradores/descomposicion_C_vs_L.png", dpi=200)
plt.show()
